# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Dataset Identifier: {metadata['identifier']}")
print(f"Published: {metadata['datePublished']}")
print(f"License: {metadata['license']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets from the dataset
record_sets = list(dataset.metadata.record_sets)
print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")

# For each record set, list fields and columns using @id
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    if rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - Field name: {getattr(f, 'name', None)}, @id: {f.id}, DataType: {getattr(f, 'data_type', None)}")
    if rs.columns:
        print("  Columns:")
        for c in rs.columns:
            print(f"    - Column name: {getattr(c, 'name', None)}, @id: {c.id}, DataType: {getattr(c, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare to extract data from all available record sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from RecordSet @id: {record_set_id}")
    if len(records) > 0:
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")

# Display first few rows for the main record set
# (Choose first non-empty record set)

main_record_set_id = None
for rs_id in record_set_ids:
    if not dataframes[rs_id].empty:
        main_record_set_id = rs_id
        break
if main_record_set_id:
    print(f"\nPreview of records from RecordSet @id {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Suppose the main record set includes GAD-7 scores and demographic variables
# Reference actual field @id's from the overview for reproducibility
# For demonstration, let us use GAD-7 score (@id: 'https://api.app.sen.science/frontiers/7160411/gad_7_score')
# and village (@id: 'https://api.app.sen.science/frontiers/7160411/village')

# Define variable IDs for analysis
numeric_field_id = 'https://api.app.sen.science/frontiers/7160411/gad_7_score'
group_field_id = 'https://api.app.sen.science/frontiers/7160411/village'

df = dataframes[main_record_set_id]

# Ensure type correctness and filter valid numeric values
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    mean_score = filtered_df[numeric_field_id].mean()
    std_score = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_score) / std_score
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by village
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean GAD-7 score):")
        display(grouped_df.head())
else:
    print(f"Numeric field '{numeric_field_id}' not found in DataFrame columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization using matplotlib and seaborn
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of GAD-7 scores
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title('Distribution of GAD-7 scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot for GAD-7 score grouped by village
    if group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title('GAD-7 Score by Village')
        plt.xlabel('Village')
        plt.ylabel('GAD-7 Score')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Summary:**
- Loaded the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using `mlcroissant`.
- Listed available record sets and fields using their `@id` identifiers for reproducibility.
- Extracted and previewed the survey responses, focusing on GAD-7 scores as a key metric.
- Demonstrated EDA including filtering records above a numeric threshold, normalization, and grouping by village.
- Visualized the distribution and variability of GAD-7 scores across villages.

**Further Steps:**
- Expand analysis to PHQ-9 and PSQ scores, demographic characteristics, and missing data patterns.
- Use field and record set `@id` references in all extraction/analysis steps for unambiguous reproducibility.
- Explore correlations between mental health scores and demographic variables.

This notebook provides a reproducible foundation for AI-ready data exploration using standardized Croissant schema referencing.